# 11 — Integrate attacking progression and defensive line geometry

**Purpose:** Join the *source-verified signed progression* output of Notebook 10 to the independently checked *static lane semantics* of Notebook 08. Produce a compact **candidate-level research table**, one record per possession start and candidate teammate, with zero invented pass-feasibility labels.

**Inputs (all local):** `outputs/P2_signed_candidate_geometry_match_{mid}.jsonl`, `outputs/P2_lane_semantics_match_{mid}.jsonl`. The signed table and lane table were independently generated. The notebook enforces one-to-one join cardinality, equal candidate identities, and consistent detection information; it stops on disagreements.

**Interpretation:** Forward endpoint displacement ≠ progressive pass; no interior projection ≠ unobstructed lane. A candidate is only a hypothetical straight-line connection between contemporaneous player locations. The outcome of any actual pass or carry is not used here. Distances are in metres. The 10 candidate records per eligible decision are correlated, not independent trial outcomes.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import json, math

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p/'data'/'matches').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Run from inside the SkillCorner repository')
OUT = ROOT/'outputs'
FILES = sorted(OUT.glob('P2_signed_candidate_geometry_match_*.jsonl'))
if len(FILES) != 20: raise AssertionError(f'Expected 20 signed-geometry files; found {len(FILES)}')

# Previous gates document reference totals for exact consistency.
signed_gate=json.loads((OUT/'P2_signed_progression_gate.json').read_text())
lane_gate=json.loads((OUT/'P2_lane_semantics_gate.json').read_text())
assert signed_gate['totals']['candidate_pairs']==lane_gate['totals']['candidate_pairs']==187570
print('Input matches:',len(FILES),'signed and lane totals:',signed_gate['totals']['candidate_pairs'])


Input matches: 20 signed and lane totals: 187570


## 1. Strict candidate-key joins

Use the complete join key `(match_id, period, frame, carrier_player_id, teammate_player_id)`. The `possession_event_id` is retained as a decision-level traceability field from the signed table. Each candidate's signed displacement and passage clearance refer to the **same** carrier and teammate. Do not merge on frame alone.

A `no_interior_projection` status refers to the defender projections measured in Notebook 08; it is neither an affirmative pass opportunity nor a guarantee of clean ball travel. `interior_projection_present` does not establish that the defender can intercept. Statuses such as `missing_opponents` and `degenerate_segment` are preserved separately.

In [2]:
def key(r):
    return (str(r['match_id']),int(r['period']),int(r['frame']),
            str(r['carrier_player_id']),str(r['teammate_player_id']))

per_match=[];totals=Counter()
for f in FILES:
    mid=f.name.split('_match_')[1].split('.jsonl')[0]
    lane_path=OUT/f'P2_lane_semantics_match_{mid}.jsonl'
    if not lane_path.is_file():raise FileNotFoundError(lane_path)
    lanes={};counter=Counter();decision_candidates=Counter()
    with lane_path.open(encoding='utf-8') as stream:
        for line in stream:
            v=json.loads(line);k=key(v)
            if k in lanes:raise AssertionError(f'{mid} duplicate lane key: {k}')
            lanes[k]=v
    seen=set()
    out=OUT/f'P2_integrated_candidate_landscape_match_{mid}.jsonl'
    with f.open(encoding='utf-8') as stream,out.open('w',encoding='utf-8') as dest:
        for line in stream:
            s=json.loads(line);k=key(s)
            if k in seen:raise AssertionError(f'{mid} duplicate signed key: {k}')
            seen.add(k)
            v=lanes.get(k)
            if v is None:raise AssertionError(f'{mid} signed candidate absent from lane file: {k}')
            if abs(float(s['pass_distance_m'])-float(v.get('pass_distance_m',s['pass_distance_m'])))>1e-7:
                raise AssertionError(f'{mid} candidate distance mismatch: {k}')
            if (s.get('carrier_detected') is True)!=(v.get('carrier_detected') is True):
                raise AssertionError(f'{mid} carrier detection disagreement: {k}')
            if s.get('teammate_quality')!=v.get('teammate_quality'):
                raise AssertionError(f'{mid} teammate quality disagreement: {k}')
            dx=float(s['signed_dx_m']);distance=float(s['pass_distance_m'])
            if not math.isfinite(dx) or not math.isfinite(distance):raise AssertionError(f'{mid} invalid distance: {k}')
            status=v['geometry_status']
            if status not in {'interior_projection_present','no_interior_projection',
                              'missing_opponents','degenerate_segment'}:
                raise AssertionError(f'{mid} unknown geometry state {status}: {k}')
            if status=='interior_projection_present' and v['interior_opponent_clearance_m'] is None:
                raise AssertionError(f'{mid} interior status with null distance: {k}')
            if status!='interior_projection_present' and v['interior_opponent_clearance_m'] is not None:
                raise AssertionError(f'{mid} unexpected distance for status {status}: {k}')
            record={
                'match_id':mid,'period':k[1],'frame':k[2],
                'possession_event_id':s.get('possession_event_id'),
                'carrier_player_id':k[3],'teammate_player_id':k[4],
                'attacking_team_id':s['attacking_team_id'],
                'source_attack_x_sign':s['source_attack_x_sign'],
                'signed_endpoint_progression_m':dx,
                'endpoint_forward_of_carrier':dx>0,
                'pass_segment_length_m':distance,
                'lateral_displacement_global_y_m':s['lateral_dy_m'],
                'geometry_status':status,
                'interior_opponent_clearance_m':v['interior_opponent_clearance_m'],
                'nearest_endpoint_defender_distance_m':v.get('nearest_endpoint_defender_distance_m'),
                'n_finite_opponents':v['n_finite_opponents'],
                'n_interior_projecting_opponents':v['n_interior_projecting_opponents'],
                'carrier_detected':s.get('carrier_detected') is True,
                'teammate_quality':s.get('teammate_quality'),
                'interpretation_level':'static_candidate_geometry_only'
            }
            dest.write(json.dumps(record,separators=(',',':'))+'\n')
            counter['pairs']+=1
            counter[f'geometry_{status}']+=1
            counter['forward_endpoint_pairs']+=int(dx>0)
            counter['nonforward_endpoint_pairs']+=int(dx<=0)
            if dx>0:counter[f'forward_and_{status}']+=1
            if s.get('carrier_detected') is True and s.get('teammate_quality')=='detected':
                counter['carrier_target_both_detected_pairs']+=1
            dk=k[:4];decision_candidates[dk]+=1
    if len(seen)!=len(lanes):
        missing=list(lanes.keys()-seen)
        raise AssertionError(f'{mid}: {len(missing)} unmatched lane rows; example {missing[:1]}')
    if any(n!=10 for n in decision_candidates.values()):
        raise AssertionError(f'{mid}: expected 10 candidates per eligible decision: {Counter(decision_candidates.values())}')
    counter['eligible_decisions']=len(decision_candidates)
    totals.update(counter)
    per_match.append({'match_id':mid,**dict(counter)})
    print(mid,'pairs',counter['pairs'],'forward',counter['forward_endpoint_pairs'],
          'no interior',counter['geometry_no_interior_projection'])


1874553 pairs 9560 forward 5407 no interior 221
1886347 pairs 9990 forward 5598 no interior 256
1899585 pairs 9290 forward 4656 no interior 187
1925299 pairs 10620 forward 5983 no interior 289
1927964 pairs 11390 forward 6417 no interior 306
1953632 pairs 9700 forward 6098 no interior 185
1959846 pairs 9450 forward 5187 no interior 193
1986691 pairs 9150 forward 4883 no interior 181
1996435 pairs 9970 forward 5347 no interior 175
1996436 pairs 7850 forward 3886 no interior 190
2006229 pairs 9770 forward 5799 no interior 173
2006363 pairs 9760 forward 5184 no interior 219
2007448 pairs 8620 forward 4784 no interior 222
2007721 pairs 9590 forward 5307 no interior 263
2010085 pairs 9350 forward 5351 no interior 209
2011166 pairs 7550 forward 3779 no interior 165
2013725 pairs 9170 forward 5076 no interior 220
2015213 pairs 8810 forward 4964 no interior 183
2016236 pairs 9590 forward 5659 no interior 199
2017461 pairs 8390 forward 4200 no interior 210


## 2. Coverage and scientific interpretation

These are **candidate-level descriptive counts** only. A football decision often contributes ten correlated candidate rows. Do not compute uncertainty or assess suppressive effectiveness from these counts as though there were 187,570 independent observations. Further work will construct episode/possession and match-level analyses, dynamic interception risk, and careful reference outcomes.

In [3]:
assert totals['pairs']==187570,totals
assert totals['eligible_decisions']==18757,totals
assert totals['forward_endpoint_pairs']==signed_gate['totals']['positive_dx']
assert totals['geometry_interior_projection_present']==lane_gate['totals']['interior_projection_present']
assert totals['geometry_no_interior_projection']==lane_gate['totals']['no_interior_projection']
assert (totals['forward_endpoint_pairs']+totals['nonforward_endpoint_pairs']==totals['pairs'])
report={
  'created_utc':datetime.now(timezone.utc).isoformat(),
  'purpose':'Join independently validated signed progression and static defender-line geometry',
  'matches':len(per_match),'totals':dict(totals),'per_match':per_match,
  'source_verification':{'signed_direction_metadata_agreements':signed_gate['metadata_goalkeeper_agreements'],
      'signed_direction_metadata_comparisons':signed_gate['metadata_goalkeeper_comparisons'],
      'lane_geometric_agreement_examples_empty':all(not x.get('disagreement_examples') for x in lane_gate['per_match'])},
  'limitations':[
     'A teammate ahead of the carrier is not necessarily an available or reachable passing option',
     'No interior defender projection is not a clear lane, viable pass, or interception-risk estimate',
     'Both carrier and teammate positions may be extrapolated; detected flags retained',
     'One decision contributes 10 candidate records; candidate rows are not independent statistical samples',
     'No match-level or possession-level outcomes or causal estimates computed',
     'No defender speeds, ball motion, receiver motion or anticipation computed',
     'No carrying alternative model or pressing coordination inferred',
     'Original match orientation metadata retained; candidate positions are static at possession start'
  ]
}
p=OUT/'P2_integrated_candidate_gate.json'
p.write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps({'matches':len(per_match),'totals':dict(totals)},indent=2))
print('Upload:',p)


{
  "matches": 20,
  "totals": {
    "pairs": 187570,
    "geometry_interior_projection_present": 183324,
    "forward_endpoint_pairs": 103565,
    "nonforward_endpoint_pairs": 84005,
    "carrier_target_both_detected_pairs": 117394,
    "forward_and_interior_projection_present": 102545,
    "geometry_no_interior_projection": 4246,
    "forward_and_no_interior_projection": 1020,
    "eligible_decisions": 18757
  }
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_integrated_candidate_gate.json
